# Five-threshold causal live development

This notebook converts the exact five-scale linkage into a causal, stateful read.

The initiating small DCC remains the immutable lineage ID. A lineage advances only through exact event-ID hand-offs:

    small seed → medium → large → macro → outer

Every live-panel row is strictly before its active stage-resolution bar. The row may contain current modes, live TMV/T/R, running progress, event counts, agreement breadth and fragmentation. It may not contain the future stage outcome or terminal depth as an actionable feature.

Outcome-conditioned tables and colours are diagnostic hindsight. Their purpose is to understand how unresolved paths develop, not to manufacture a trading label.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events
from dc_basics.data import resolve_data_root

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160, 320, 640, 1280]
MIN_AUC_CLASS_COUNT = 20
# Checkpoints at which live geometry is compared with completed initiation features.
INCREMENTAL_CHECKPOINTS = [0, 10, 20, 40, 80]
CASEBOOK_BARS_BEFORE = 180
CASEBOOK_BARS_AFTER = 180

In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"
    OUTER = "outer"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"
    LARGE_TO_MACRO = "large_to_macro"
    MACRO_TO_OUTER = "macro_to_outer"


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    STOPPED_BELOW_MACRO = "stopped_below_macro"
    STOPPED_BELOW_OUTER = "stopped_below_outer"
    REACHED_OUTER = "reached_outer"
    AMBIGUOUS_STAGE = "ambiguous_stage"
    UNRESOLVED_STAGE = "unresolved_stage"
    UNLINKED_HANDOFF = "unlinked_handoff"


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    COUNTER_DEPTH_1 = "counter_depth_1"
    COUNTER_DEPTH_2 = "counter_depth_2"
    COUNTER_DEPTH_3 = "counter_depth_3"
    COUNTER_DEPTH_4 = "counter_depth_4"
    FRAGMENTED_2_BOUNDARIES = "fragmented_2_boundaries"
    FRAGMENTED_3_BOUNDARIES = "fragmented_3_boundaries"
    FRAGMENTED_4_BOUNDARIES = "fragmented_4_boundaries"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO, Scale.OUTER)
THRESHOLDS = {
    Scale.SMALL: 0.002,
    Scale.MEDIUM: 0.003,
    Scale.LARGE: 0.0045,
    Scale.MACRO: 0.00675,
    Scale.OUTER: 0.010125,
}
SCALE_PREFIXES = {
    Scale.SMALL: "S",
    Scale.MEDIUM: "M",
    Scale.LARGE: "L",
    Scale.MACRO: "X",
    Scale.OUTER: "O",
}
STAGE_ORDER = (
    Stage.SMALL_TO_MEDIUM,
    Stage.MEDIUM_TO_LARGE,
    Stage.LARGE_TO_MACRO,
    Stage.MACRO_TO_OUTER,
)
STAGE_SPECS = {
    Stage.SMALL_TO_MEDIUM: (Scale.SMALL, Scale.MEDIUM),
    Stage.MEDIUM_TO_LARGE: (Scale.MEDIUM, Scale.LARGE),
    Stage.LARGE_TO_MACRO: (Scale.LARGE, Scale.MACRO),
    Stage.MACRO_TO_OUTER: (Scale.MACRO, Scale.OUTER),
}
STOPPED_OUTCOMES = {
    0: TerminalOutcome.STOPPED_BELOW_MEDIUM,
    1: TerminalOutcome.STOPPED_BELOW_LARGE,
    2: TerminalOutcome.STOPPED_BELOW_MACRO,
    3: TerminalOutcome.STOPPED_BELOW_OUTER,
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

## Stateful information contract

The lineage is idle until an eligible small DCC leaves five-scale broad agreement. During a stage:

- the lower mode points in the target direction;
- the upper mode still points in the origin direction;
- target progress measures the causal reversal from the active upper-scale running extreme;
- attained depth is already known;
- future stage outcome and terminal depth remain unknown.

The resolution bar is excluded because the outcome becomes known on that bar.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Symbol: {SYMBOL}")
print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Symbol: GBPUSD.r
Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger as _prepare_event_ledger

def prepare_event_ledger(source: pd.DataFrame, scale: Scale, theta: float) -> pd.DataFrame:
    return _prepare_event_ledger(source, SCALE_PREFIXES[scale], scale, theta)


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    theta = THRESHOLDS[scale]
    ledger = prepare_event_ledger(parse_dc_events(prices=prices, theta=theta), scale, theta)
    live = dc_live_features(prices, theta=theta)
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(
        ledger.set_index("dcc_idx")["event_id"]
    )

## Reconstruct the exact four-stage lineage

The same adjacency and exact-hand-off rules used in the linkage notebook are repeated here so that the live notebook is independently executable.

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


attempts_by_stage = {}
for stage in STAGE_ORDER:
    lower_scale, upper_scale = STAGE_SPECS[stage]
    attempts = build_adjacent_attempts(mode_frame, lower_scale, upper_scale)
    attempts.insert(0, "stage", stage.value)
    attempts_by_stage[stage] = attempts

mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
previous_modes = mode_frame[mode_columns].shift(1)
small_attempts = attempts_by_stage[Stage.SMALL_TO_MEDIUM]
seed_times = pd.DatetimeIndex(small_attempts["start_time"])
seed_previous_modes = previous_modes.loc[seed_times]
eligible_seed_mask = (
    seed_previous_modes.notna().all(axis=1)
    & seed_previous_modes.nunique(axis=1).eq(1)
).to_numpy()
seeds = small_attempts.loc[eligible_seed_mask].copy()

In [6]:
from dc_basics.lineage import build_five_scale_linkages


linkages, linkage_stages = build_five_scale_linkages(seeds, attempts_by_stage)
strict_clean_linkages = linkages.loc[linkages["strict_clean"]].copy()
strict_ids = set(strict_clean_linkages["lineage_id"])
strict_clean_stages = linkage_stages.loc[linkage_stages["lineage_id"].isin(strict_ids)].copy()

if not strict_clean_stages["resolution_bars"].gt(0).all():
    raise AssertionError("A strict-clean stage has no pre-resolution bars.")

print(f"Eligible seeds: {len(linkages):,}")
print(f"Strict-clean lineages: {len(strict_clean_linkages):,}")
print(f"Strict-clean traversed stages: {len(strict_clean_stages):,}")

Eligible seeds: 799
Strict-clean lineages: 751
Strict-clean traversed stages: 1,753


## Causal hierarchy geometry

Running extremes reset only when their own live mode changes. Progress is the signed reversal from the active upper-scale extreme divided by that upper threshold. It must remain below one on every pre-resolution row.

In [7]:
from dc_basics.configuration import classify_configuration as _classify_configuration

def classify_configuration(modes: tuple[str, ...]) -> dict[str, object]:
    return _classify_configuration(modes, outer_scale_name="outer", expected_count=5)


all_modes_known = mode_frame[mode_columns].notna().all(axis=1)
configuration_properties = pd.DataFrame.from_records(
    [
        classify_configuration(tuple(row))
        for row in mode_frame.loc[all_modes_known, mode_columns].itertuples(index=False, name=None)
    ],
    index=mode_frame.index[all_modes_known],
)
for column in configuration_properties.columns:
    mode_frame.loc[configuration_properties.index, column] = configuration_properties[column]


from dc_basics.live import build_causal_running_extreme


running_extremes = {
    scale: build_causal_running_extreme(prices, live_frames[scale])
    for scale in SCALE_ORDER
}
dcc_cumulative = {}
for scale in SCALE_ORDER:
    dcc_flag = (
        mode_frame[f"{scale.value}_upward_dcc"]
        | mode_frame[f"{scale.value}_downward_dcc"]
    ).astype(np.int64)
    dcc_cumulative[scale] = dcc_flag.cumsum().to_numpy(dtype=np.int64)

In [8]:
from dc_basics.audit import audit_live_panel
from dc_basics.live_panel import LivePanelContext, build_live_panel


live_context = LivePanelContext(
    prices=prices,
    mode_frame=mode_frame,
    live_frames=live_frames,
    running_extremes=running_extremes,
    thresholds=THRESHOLDS,
    dcc_cumulative=dcc_cumulative,
    scale_order=SCALE_ORDER,
    hierarchy_columns=(
        "configuration", "hierarchy_shape", "direction_boundaries",
        "outer_agreement_breadth", "counter_depth", "coherent_frontier",
    ),
    event_ledgers=event_ledgers,
)
live_panel = build_live_panel(strict_clean_stages, strict_clean_linkages, live_context, STAGE_SPECS)


In [9]:
expected_stage_counts = {
    stage.value: int(strict_clean_linkages["terminal_depth"].ge(stage_index).sum())
    for stage_index, stage in enumerate(STAGE_ORDER)
}
live_panel_audit = audit_live_panel(
    live_panel,
    expected_stage_counts=expected_stage_counts,
    required_features=(
        "lower_tmv_live", "lower_t_live", "lower_r_live",
        "upper_tmv_live", "upper_t_live", "upper_r_live",
        "target_progress", "max_target_progress", "progress_drawdown",
    ),
)
display(live_panel_audit.to_frame())

# The active lower and upper clocks never fire inside a stage panel (the audit's
# maximum_nested_* rows show this for each clock). Activity below the lower clock
# is the informative nested count; Stage 1 has no clock below small.
live_panel.groupby("stage", sort=False).agg(
    sub_lower_scales=("sub_lower_scale_count", "first"),
    maximum_sub_lower_dccs=("sub_lower_dccs", "max"),
    maximum_lineage_age_bars=("lineage_age_bars", "max"),
)


,value
lineages,751.000000
active_pre_resolution_rows,179087.000000
maximum_pre_resolution_progress,0.999962
seed_ids_that_changed,0.000000
large_to_macro_lineages,317.000000
macro_to_outer_lineages,196.000000
medium_to_large_lineages,489.000000
small_to_medium_lineages,751.000000
maximum_nested_small_dccs,16.000000
maximum_nested_medium_dccs,9.000000


## Information-timing audit

All fields above the diagnostic divider are available on the feature bar. Outcome columns are retained only to compare later-resolved groups after the fact.

In [10]:
information_contract = pd.DataFrame([
    ("lineage_id", True, "immutable initiating small event ID"),
    ("stage_start_event_id", True, "exact confirmed hand-off event"),
    ("attained_depth", True, "largest scale already confirmed"),
    ("elapsed_bars / lineage_age_bars", True, "bars since the stage start / since the immutable seed"),
    ("configuration / hierarchy_shape", True, "current five-mode structure"),
    ("direction_boundaries", True, "current scale fragmentation"),
    ("outer_agreement_breadth", True, "number of modes agreeing with outer"),
    ("target_progress", True, "current fraction of active upper threshold"),
    ("max_target_progress / drawdown", True, "causal path history inside the active stage"),
    ("lower and upper TMV/T/R", True, "live geometry of active adjacent clocks"),
    ("seed_leg_* / stage_start_leg_*", True, "completed prior-leg TMV/T/R confirmed by the seed / hand-off DCC"),
    ("sub_lower_dccs / sub_lower_dcc_rate", True, "DCCs since stage start on clocks below the active lower clock"),
    ("future_stage_outcome", False, "known only on stage-resolution DCC"),
    ("future_terminal_depth", False, "known only when complete lineage terminates"),
], columns=["column", "known_on_feature_bar", "economic_meaning"])
information_contract

,column,known_on_feature_bar,economic_meaning
0,lineage_id,True,immutable initiating small event ID
1,stage_start_event_id,True,exact confirmed hand-off event
2,attained_depth,True,largest scale already confirmed
3,configuration / hierarchy_shape,True,current five-mode structure
4,direction_boundaries,True,current scale fragmentation
5,outer_agreement_breadth,True,number of modes agreeing with outer
6,target_progress,True,current fraction of active upper threshold
7,max_target_progress / drawdown,True,causal path history inside the active stage
8,lower and upper TMV/T/R,True,live geometry of active adjacent clocks
9,nested DCC counts and rate,True,confirmed activity since stage start


## Live Audit 1: fixed checkpoints and survival conditioning

At checkpoint k, only attempts unresolved after k bars survive. Later checkpoint rows are selected populations and cannot be compared as if they were the initiation cohort.

In [11]:
checkpoint_rows = live_panel.loc[live_panel["elapsed_bars"].isin(CHECKPOINT_BARS)].copy()
stage_attempt_counts = live_panel.groupby("stage")["lineage_id"].nunique()

checkpoint_survival = checkpoint_rows.groupby(["stage", "elapsed_bars"], sort=True).agg(
    surviving_attempts=("lineage_id", "nunique"),
    later_propagation_rate=(
        "future_stage_outcome",
        lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean(),
    ),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    progress_drawdown_median=("progress_drawdown", "median"),
    lower_r_median=("lower_r_live", "median"),
    direction_boundaries_median=("direction_boundaries", "median"),
    outer_agreement_breadth_median=("outer_agreement_breadth", "median"),
    sub_lower_dccs_median=("sub_lower_dccs", "median"),
    sub_lower_dcc_rate_median=("sub_lower_dcc_rate", "median"),
).reset_index()
checkpoint_survival["survivor_share"] = checkpoint_survival.apply(
    lambda row: row["surviving_attempts"] / stage_attempt_counts[row["stage"]],
    axis=1,
)

display(checkpoint_survival.set_index(["stage", "elapsed_bars"]).style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "progress_drawdown_median": "{:.2%}",
    "lower_r_median": "{:.3e}",
    "direction_boundaries_median": "{:.1f}",
    "outer_agreement_breadth_median": "{:.1f}",
    "sub_lower_dccs_median": "{:.1f}",
    "sub_lower_dcc_rate_median": "{:.3f}",
    "survivor_share": "{:.2%}",
}))

In [12]:
checkpoint_outcomes = checkpoint_rows.groupby(
    ["stage", "elapsed_bars", "future_stage_outcome", "target_mode"], sort=True
).agg(
    surviving_attempts=("lineage_id", "size"),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    progress_drawdown_median=("progress_drawdown", "median"),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    max_favorable_excursion_median=("max_favorable_excursion_theta", "median"),
    max_adverse_excursion_median=("max_adverse_excursion_theta", "median"),
    direction_boundaries_median=("direction_boundaries", "median"),
    outer_agreement_breadth_median=("outer_agreement_breadth", "median"),
    sub_lower_dccs_median=("sub_lower_dccs", "median"),
)
checkpoint_outcomes.style.format({
    "surviving_attempts": "{:,}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "progress_drawdown_median": "{:.2%}",
    "lower_tmv_median": "{:.3f}",
    "lower_r_median": "{:.3e}",
    "max_favorable_excursion_median": "{:.3f}",
    "max_adverse_excursion_median": "{:.3f}",
    "direction_boundaries_median": "{:.1f}",
    "outer_agreement_breadth_median": "{:.1f}",
    "sub_lower_dccs_median": "{:.1f}",
})

## Live Audit 2: outcome overlap

Orientation-free AUC measures univariate ordering between later propagated and rejected survivors. Every feature value is causal; using the later outcome to calculate AUC makes the analysis diagnostic, not actionable. AUC near 0.5 means substantial overlap.

Each table shows the propagated and rejected survivor counts beside the AUCs. Cells without an AUC say why: `insufficient_sample` (fewer than `MIN_AUC_CLASS_COUNT` survivors in a group), `constant_by_construction` (the feature cannot vary inside the panel, such as Stage 1 sub-lower activity, because no clock lies below small) or `constant_in_sample`.

In Stages 1 and 2, direction-boundary count and outer-agreement breadth are deterministic functions of each other: both are constant in Stage 1, and in Stage 2 both are fixed by the small mode alone. Identical AUC pairs there are arithmetic, not evidence. Compare the two measures from Stage 3 on.

In [13]:
from dc_basics.live_panel import (
    auc_table_with_counts,
    constant_by_construction_features,
    summarize_checkpoint_overlap,
)

LIVE_FEATURE_COLUMNS = [
    "lower_tmv_live",
    "lower_t_live",
    "lower_r_live",
    "upper_tmv_live",
    "upper_t_live",
    "upper_r_live",
    "target_progress",
    "max_target_progress",
    "progress_drawdown",
    "max_favorable_excursion_theta",
    "max_adverse_excursion_theta",
    "direction_boundaries",
    "outer_agreement_breadth",
    "sub_lower_dccs",
    "sub_lower_dcc_rate",
    "lineage_age_bars",
]
CONSTANT_FEATURES = constant_by_construction_features(STAGE_SPECS, SCALE_ORDER)

overlap_summary = summarize_checkpoint_overlap(
    checkpoint_rows,
    LIVE_FEATURE_COLUMNS,
    min_class_count=MIN_AUC_CLASS_COUNT,
    constant_by_construction=CONSTANT_FEATURES,
)
selected_features = [
    "target_progress", "lower_r_live", "progress_drawdown",
    "direction_boundaries", "outer_agreement_breadth", "sub_lower_dcc_rate",
]
for stage in STAGE_ORDER:
    print(stage.value)
    display(auc_table_with_counts(
        overlap_summary.loc[overlap_summary["stage"].eq(stage.value)],
        features=selected_features,
    ))

small_to_medium


feature,direction_boundaries,lower_r_live,nested_lower_dcc_rate,outer_agreement_breadth,progress_drawdown,target_progress
elapsed_bars,,,,,,
0,nan,0.510,nan,nan,nan,0.548
5,nan,0.548,nan,nan,0.684,0.691
10,nan,0.592,nan,nan,0.721,0.734
20,nan,0.663,nan,nan,0.778,0.788
40,nan,0.669,nan,nan,0.717,0.745
80,nan,0.668,nan,nan,0.718,0.699
160,nan,nan,nan,nan,nan,nan
320,nan,nan,nan,nan,nan,nan


medium_to_large


feature,direction_boundaries,lower_r_live,nested_lower_dcc_rate,outer_agreement_breadth,progress_drawdown,target_progress
elapsed_bars,,,,,,
0,nan,0.505,nan,nan,nan,0.589
5,0.509,0.512,nan,0.509,0.590,0.630
10,0.514,0.532,nan,0.514,0.621,0.664
20,0.550,0.580,nan,0.550,0.660,0.694
40,0.581,0.601,nan,0.581,0.627,0.654
80,0.592,0.672,nan,0.592,0.697,0.695
160,0.646,0.755,nan,0.646,0.728,0.762
320,nan,nan,nan,nan,nan,nan
640,nan,nan,nan,nan,nan,nan


large_to_macro


feature,direction_boundaries,lower_r_live,nested_lower_dcc_rate,outer_agreement_breadth,progress_drawdown,target_progress
elapsed_bars,,,,,,
0,nan,0.568,nan,nan,nan,0.605
5,0.512,0.574,nan,0.512,0.610,0.650
10,0.539,0.579,nan,0.539,0.632,0.687
20,0.588,0.590,nan,0.590,0.651,0.704
40,0.575,0.629,nan,0.589,0.649,0.704
80,0.606,0.653,nan,0.638,0.685,0.726
160,0.597,0.672,nan,0.591,0.637,0.717
320,nan,nan,nan,nan,nan,nan
640,nan,nan,nan,nan,nan,nan


macro_to_outer


feature,direction_boundaries,lower_r_live,nested_lower_dcc_rate,outer_agreement_breadth,progress_drawdown,target_progress
elapsed_bars,,,,,,
0,nan,0.501,nan,nan,nan,0.514
5,0.503,0.502,nan,0.503,0.515,0.533
10,0.509,0.505,nan,0.510,0.575,0.561
20,0.556,0.535,nan,0.563,0.658,0.639
40,0.622,0.575,nan,0.642,0.704,0.717
80,0.671,0.586,nan,0.675,0.666,0.690
160,0.639,0.629,nan,0.675,0.710,0.753
320,0.547,0.741,nan,0.664,0.714,0.724
640,nan,nan,nan,nan,nan,nan


## Live Audit 2b: is the separation directionally consistent?

The pooled AUC above mixes up and down targets. A structural relationship should point the same way for both. Here the same overlap diagnostic is computed separately for each target mode. A (stage, checkpoint, feature) cell is **testable** only when both directions have an adequate sample, and **consistent** when both raw AUCs fall on the same side of 0.5.

`raw_auc_for_higher_value` above 0.5 means higher values go with later propagation. Every feature here is already direction-neutral (progress and excursions are signed toward the target; TMV, T, R and counts are unsigned), so the up/down comparison is like for like.

In [ ]:
direction_overlap = summarize_checkpoint_overlap(
    checkpoint_rows,
    LIVE_FEATURE_COLUMNS,
    group_columns=("stage", "elapsed_bars", "target_mode"),
    min_class_count=MIN_AUC_CLASS_COUNT,
    constant_by_construction=CONSTANT_FEATURES,
)
direction_wide = direction_overlap.pivot_table(
    index=["stage", "elapsed_bars", "feature"],
    columns="target_mode",
    values=["raw_auc_for_higher_value", "propagated_survivors", "rejected_survivors"],
    aggfunc="first",
    dropna=False,
)
direction_wide.columns = [f"{value}_{mode}" for value, mode in direction_wide.columns]
direction_wide = direction_wide.reset_index()
up_auc = direction_wide["raw_auc_for_higher_value_up"]
down_auc = direction_wide["raw_auc_for_higher_value_down"]
direction_wide["testable"] = up_auc.notna() & down_auc.notna()
direction_wide["consistent"] = direction_wide["testable"] & (up_auc.ge(0.5) == down_auc.ge(0.5))

direction_consistency = direction_wide.groupby(["stage", "feature"], sort=False).agg(
    testable_checkpoints=("testable", "sum"),
    consistent_checkpoints=("consistent", "sum"),
)
direction_consistency = direction_consistency.loc[direction_consistency["testable_checkpoints"].gt(0)]
display(direction_consistency)

direction_wide.loc[
    direction_wide["testable"] & direction_wide["feature"].isin(selected_features)
].set_index(["stage", "elapsed_bars", "feature"]).style.format({
    "raw_auc_for_higher_value_up": "{:.3f}",
    "raw_auc_for_higher_value_down": "{:.3f}",
}, precision=0)

## Live Audit 2c: live geometry versus completed prior-leg geometry

Live TMV/T/R describe an unfinished path. The **completed** prior-leg TMV/T/R confirmed by the seed DCC (`seed_leg_*`) and by the stage's hand-off DCC (`stage_start_leg_*`) were already known when the stage began. Both sets are compared on the **same survivor cohort** at each checkpoint.

`best_*_auc` is the best univariate in-sample AUC of each set. The `oos_auc_*` columns come from logistic models scored with forward-chaining folds ordered by stage start, so each attempt is scored only by a model fitted on earlier attempts. A positive `oos_auc_gain` says the live path carries information the completed initiation legs did not. Deep stages have few lineages, so many cells are `insufficient_sample`.

In [ ]:
from dc_basics.live_panel import compare_initiation_and_live_features

SEED_LEG_FEATURES = ["seed_leg_tmv", "seed_leg_t", "seed_leg_r"]
STAGE_LEG_FEATURES = ["stage_start_leg_tmv", "stage_start_leg_t", "stage_start_leg_r"]
LIVE_COMPARISON_FEATURES = [
    "target_progress", "max_target_progress", "lower_tmv_live", "lower_r_live",
    "max_favorable_excursion_theta", "max_adverse_excursion_theta",
]

incremental_records = []
for stage in STAGE_ORDER:
    is_first_stage = stage == Stage.SMALL_TO_MEDIUM
    initiation_features = SEED_LEG_FEATURES + ([] if is_first_stage else STAGE_LEG_FEATURES)
    live_features = LIVE_COMPARISON_FEATURES + (
        [] if is_first_stage else ["direction_boundaries", "sub_lower_dccs"]
    )
    for elapsed_bars in INCREMENTAL_CHECKPOINTS:
        cohort = checkpoint_rows.loc[
            checkpoint_rows["stage"].eq(stage.value) & checkpoint_rows["elapsed_bars"].eq(elapsed_bars)
        ]
        incremental_records.append({
            "stage": stage.value,
            "elapsed_bars": elapsed_bars,
            **compare_initiation_and_live_features(
                cohort, initiation_features, live_features, min_class_count=MIN_AUC_CLASS_COUNT,
            ),
        })

incremental_summary = pd.DataFrame.from_records(incremental_records).set_index(["stage", "elapsed_bars"])
incremental_summary.style.format({
    "best_initiation_auc": "{:.3f}",
    "best_live_auc": "{:.3f}",
    "oos_auc_initiation": "{:.3f}",
    "oos_auc_initiation_plus_live": "{:.3f}",
    "oos_auc_gain": "{:+.3f}",
}, na_rep="—")

## Live Audit 3: survival-conditioned trajectories

Blue denotes later propagation and orange later rejection. Solid and dashed lines distinguish target direction. The line at progress one is the upper DCC boundary and should not be crossed inside the pre-resolution panel.

The fourth panel shows maximum adverse excursion in Stage 1 and, from Stage 2 on, the DCC rate on clocks below the active lower clock. The active lower clock's own DCC count is zero by construction inside the panel, so it is not plotted.

In [14]:
from dc_basics.live_plotting import plot_stage_trajectories as _plot_stage_trajectories

def plot_stage_trajectories(panel: pd.DataFrame, stage: Stage, max_elapsed_bars: int) -> go.Figure:
    # Library defaults: Stage 1 plots max adverse excursion (nothing fires below the small clock);
    # later stages plot the DCC rate on clocks below the active lower clock.
    return _plot_stage_trajectories(panel, stage, max_elapsed_bars, symbol=SYMBOL)


stage_plot_horizons = {
    Stage.SMALL_TO_MEDIUM: 160,
    Stage.MEDIUM_TO_LARGE: 320,
    Stage.LARGE_TO_MACRO: 640,
    Stage.MACRO_TO_OUTER: 1280,
}
for stage in STAGE_ORDER:
    plot_stage_trajectories(live_panel, stage, stage_plot_horizons[stage]).show()

## Live Audit 4: hierarchy-shape occupancy

Current shape is causal. Conditioning shape occupancy on the later outcome is hindsight and asks whether coherent versus fragmented paths develop differently while unresolved.

In [15]:
shape_occupancy = pd.crosstab(
    [
        checkpoint_rows["stage"],
        checkpoint_rows["elapsed_bars"],
        checkpoint_rows["future_stage_outcome"],
    ],
    checkpoint_rows["hierarchy_shape"],
    normalize="index",
)
shape_occupancy.style.format("{:.2%}")

## Five-depth live casebook

One recent strict-clean lineage is selected for each observed terminal depth. Coloured price segments contain only pre-resolution feature bars. Handoff and terminal markers are displayed after the fact to explain where information became knowable.

In [16]:
casebook_lineages = (
    strict_clean_linkages.sort_values("seed_time")
    .groupby("terminal_depth", sort=True, group_keys=False)
    .tail(1)
    .sort_values("terminal_depth")
)
display(casebook_lineages[[
    "lineage_id", "seed_time", "target_mode",
    "terminal_time", "terminal_depth", "terminal_outcome",
]])

stage_colors = {
    Stage.SMALL_TO_MEDIUM.value: "#16a34a",
    Stage.MEDIUM_TO_LARGE.value: "#2563eb",
    Stage.LARGE_TO_MACRO.value: "#7c3aed",
    Stage.MACRO_TO_OUTER.value: "#ec4899",
}

,lineage_id,seed_time,target_mode,terminal_time,terminal_depth,terminal_outcome
797,S2695,2026-11-06 17:20:00+02:00,up,2026-11-06 21:45:00+02:00,0,stopped_below_medium
795,S2690,2026-11-04 17:00:00+02:00,down,2026-11-05 06:10:00+02:00,1,stopped_below_large
787,S2660,2026-10-16 17:25:00+02:00,down,2026-10-20 08:25:00+02:00,2,stopped_below_macro
785,S2649,2026-10-08 14:25:00+02:00,up,2026-10-14 15:50:00+02:00,3,stopped_below_outer
796,S2692,2026-11-05 09:30:00+02:00,down,2026-11-06 15:25:00+02:00,4,reached_outer


In [17]:
def plot_live_casebook(
    close: pd.Series, panel: pd.DataFrame, examples: pd.DataFrame, stages: pd.DataFrame
) -> go.Figure:
    figure = make_subplots(
        rows=len(examples),
        cols=1,
        vertical_spacing=min(0.04, 0.15 / len(examples)),
        subplot_titles=[
            f"depth {int(example.terminal_depth)} | target={example.target_mode} | seed={example.lineage_id}"
            for example in examples.itertuples(index=False)
        ],
    )

    for row_number, example in enumerate(examples.itertuples(index=False), start=1):
        window_start = max(0, int(example.seed_position) - CASEBOOK_BARS_BEFORE)
        window_end = min(len(close), int(example.terminal_position) + CASEBOOK_BARS_AFTER + 1)
        price_window = close.iloc[window_start:window_end]
        lineage_path = panel.loc[panel["lineage_id"].eq(example.lineage_id)]
        lineage_stages = stages.loc[stages["lineage_id"].eq(example.lineage_id)].sort_values("stage_order")

        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#94a3b8", "width": 1.0},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)

        for stage, stage_path in lineage_path.groupby("stage", sort=False):
            figure.add_trace(go.Scattergl(
                x=stage_path["time"],
                y=stage_path["price"],
                mode="lines",
                name=stage,
                legendgroup=stage,
                showlegend=row_number == 1,
                line={"color": stage_colors[stage], "width": 3.0},
                customdata=stage_path[[
                    "elapsed_bars", "target_progress", "lower_r_live",
                    "direction_boundaries", "outer_agreement_breadth",
                    "sub_lower_dccs", "lineage_age_bars",
                ]].to_numpy(),
                hovertemplate=(
                    f"{stage}<br>time=%{{x}}<br>close=%{{y:.5f}}<br>"
                    "elapsed=%{customdata[0]:.0f}<br>progress=%{customdata[1]:.2%}<br>"
                    "live lower R=%{customdata[2]:.3e}<br>"
                    "boundaries=%{customdata[3]:.0f}<br>"
                    "outer breadth=%{customdata[4]:.0f}<br>"
                    "DCCs below lower clock=%{customdata[5]:.0f}<br>"
                    "lineage age=%{customdata[6]:.0f}<extra></extra>"
                ),
            ), row=row_number, col=1)

        for stage_row in lineage_stages.itertuples(index=False):
            marker_time = stage_row.start_time
            marker_price = close.iloc[int(stage_row.start_position)]
            figure.add_trace(go.Scatter(
                x=[marker_time],
                y=[marker_price],
                mode="markers",
                name=f"{stage_row.stage} start",
                legendgroup=f"{stage_row.stage}_start",
                showlegend=row_number == 1,
                marker={"color": stage_colors[stage_row.stage], "size": 8, "symbol": "circle"},
                hovertemplate=(
                    f"{stage_row.stage} start<br>time=%{{x}}<br>"
                    "price=%{y:.5f}<extra></extra>"
                ),
            ), row=row_number, col=1)

        terminal_price = close.iloc[int(example.terminal_position)]
        figure.add_trace(go.Scatter(
            x=[example.terminal_time],
            y=[terminal_price],
            mode="markers",
            name="Terminal resolution",
            legendgroup="terminal",
            showlegend=row_number == 1,
            marker={"color": "#111827", "size": 10, "symbol": "diamond"},
            hovertemplate="terminal resolution<br>time=%{x}<br>price=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: causal five-scale live-development casebook",
        template="plotly_white",
        height=max(1000, 330 * len(examples)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.04, "x": 0.0},
    )
    figure.update_xaxes(title="Time", row=len(examples), col=1)
    return figure


plot_live_casebook(prices, live_panel, casebook_lineages, strict_clean_stages).show()

# Five-threshold causal live-development completion gate

Before robustness, be able to explain:

1. Why the immutable small seed survives every confirmed hand-off.
2. Why attained depth is causal but future terminal depth is not.
3. Why each panel row stops one bar before its stage resolution.
4. Why target progress uses the active upper-scale running extreme.
5. Why progress must stay below one inside the panel.
6. Why the lower mode remains target while the upper mode remains origin.
7. Why live TMV/T/R differ from completed prior-leg TMV/T/R, and whether the live path adds information beyond them (Live Audit 2c).
8. Why the active lower and upper clocks cannot fire inside a stage panel, and why DCC counts and firing rates on the clocks below the lower clock are the informative nested activity.
9. Why agreement breadth and direction-boundary count provide different information.
10. Why later checkpoints contain selected survivors.
11. Why a changing later-propagation rate does not imply that waiting causes propagation.
12. Why outcome-conditioned AUC is diagnostic even though its input features are causal.
13. Which live features gain separation only after a path develops.
14. Whether that separation is directionally consistent for up and down targets (Live Audit 2b).
15. Why a depth-four path confirms outer propagation but does not forecast continuation.
16. Why none of these audits yet establishes economic profitability.

The next step is predeclared structural robustness across symbols, absolute threshold scales, calendar periods, directions and the linear-grid sensitivity benchmark.